# 11 - What the state cost

**Purpose.** Session 04 established that the black-level state is analog and sits *before* the
gain stage (`offset_state_mechanism` = H2). This notebook asks what that verdict costs the
constants published before it, and it does so **entirely from frames already on disk**. Three
questions, ordered by how far they move a number the model consumes:

1. **`g(gain)`** - session 02's PTC subtracted one pedestal per gain, and that pedestal was the
   mean of ten bias frames that were themselves a state mixture. The PTC's intercept was *fixed*
   at session 01's `R^2` rather than fitted, so every error in the signal axis lands in the slope.
   How far does `system_gain` move once flats and bias are differenced in the same state?
2. **The gain-100 bimodality** - session 04's strongest, out-of-sample confirmation of H2 lives in
   a scratchpad and in prose. Publish it through `stats.offset_state` with provenance, and test
   whether the state is large enough to be L31's missing mechanism.
3. **`D`** - session 03 published an upper bound and named the state as what limited it. Test that
   claim. The peer-group rule can only see hopping *within* a group of frames that share a
   setting; if the offset tracks exposure length, the rule is blind to it by construction.

**What it is not for.** No camera, no new frames, and the `Z:` archive is untouched. It does not
settle gain 200, which needs the camera. It does not re-run session 02's capture: the flats and
bias are re-read, and `results/ptc_rungs.csv` supplies the variance axis unchanged, because the
pair-difference variance is a width in space and the state is a level - session 04's `R` result
is the reason that axis needs no repair.

**Where its numbers land.** `results/state_repair_constants.json`, this notebook's own file. It
deliberately does **not** overwrite `system_gain` in `ptc_constants.json`. Whether a corrected
constant supersedes in place or lives beside the original is a decision about the record, taken
in conversation after review - not by the notebook that found the correction.

**It assumes `00_statistics.ipynb`** for why a plane mean over a quarter of a million pixels
resolves a hundredth of a count, and `09`/`10` for the state itself.

In [ ]:
import csv, json, itertools, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from astropix import fits, spatial, stats

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, RESULTS = REPO / "data", REPO / "results"
PLANES = ("R", "G1", "G2", "B")
GAINS = [0, 50, 100, 190, 200, 250, 300, 450]

published = lambda n: json.loads((RESULTS / n).read_text())
table = lambda n: list(csv.DictReader((RESULTS / n).open()))

bias_c, ptc_c = published("bias_constants.json"), published("ptc_constants.json")
dark_c, state_c = published("dark_constants.json"), published("offset_state_constants.json")
rungs, drift, blocks = table("ptc_rungs.csv"), table("pedestal_drift.csv"), table("dark_blocks.csv")

# The H2 law, rebuilt from published constants rather than retyped.  It is used only to
# *predict* a step size, never to detect one: detection is stats.offset_state, always.
PED = bias_c["pedestal_fit"]["value"]
ANCHOR = dark_c["offset_state_step"]["value"]
_k = ANCHOR / (PED["hcg"]["B"] * 10 ** (250 / 200))
h2_step = lambda g: (PED["lcg"] if g < bias_c["hcg_threshold_gain"]["value"]
                     else PED["hcg"])["B"] * 10 ** (g / 200) * _k

print("H2 predicted step, ADC counts")
print("  " + "  ".join("g%d %.3f" % (g, h2_step(g)) for g in GAINS))
print("\nsession 02 published g:", {g: ptc_c["system_gain"]["value"][str(g)] for g in GAINS})

---

## 1. Gate 1 before anything else

`CLAUDE.md` makes the white-balance proof gate 1 of every bench protocol, and the proof is in the
pixels rather than in the control: the modal step between adjacent distinct values must be **16 on
all four planes**. These frames passed that gate on the night they were shot, in session 02's own
notebook. Re-running it here costs one frame per gain and is the difference between *trusting a
record* and *checking one* - and this notebook exists because a record turned out to need checking.

`stats.to_adc` refuses rather than truncates, so it is the same gate a second time: a frame whose
low bits are set never reaches the arithmetic below.

In [ ]:
def plane_levels(path):
    'One frame -> its four plane means in ADC counts, and its header.'
    mosaic, header = fits.read(path)
    planes = spatial.split(stats.to_adc(mosaic))
    return {p: float(planes[p].mean()) for p in PLANES}, header


gate1 = []
for g in GAINS:
    probe = sorted((DATA / "session02" / "frames").glob("bias_g%03d_*.fits" % g))[0]
    mosaic, _ = fits.read(probe)
    steps = {p: stats.value_step(v) for p, v in spatial.split(mosaic).items()}
    gate1.append((g, steps, all(s == 16 for s in steps.values())))

for g, steps, ok in gate1:
    print("gain %3d  %s  %s" % (g, steps, "PASS" if ok else "*** FAIL ***"))
assert all(ok for _, _, ok in gate1), "gate 1: white balance is still applied"

## 2. The bias axis: what session 02's pedestals actually were

Ten bias frames per gain, at offset 15, minimum exposure, in the session's ROI. These are the
frames whose mean became the `pedestal` column of `ptc_rungs.csv` - identical down all twelve rungs
of a gain, which is how we know it was taken once per gain and not per rung.

Bias frames carry no light, so there is no source drift to confuse with a state, and the classifier
is on its home ground. What comes back per gain is the realised **far fraction** `f_b`, and with it
the systematic part of the pedestal error: a mixture mean sits `f_b * step` above the near state.

The prediction to check against is not free: `offset_state_occupancy` measured 33.3% at gain 0 and
25.0% at gain 450 on a different night, and section 6 of `10` backed out 20.0% from session 01's
published pedestal at gain 450. A third independent number from a third night is worth having.

In [ ]:
bias = {}
for g in GAINS:
    paths = sorted((DATA / "session02" / "frames").glob("bias_g%03d_*.fits" % g))
    per_frame = [plane_levels(p)[0] for p in paths]
    per_plane = {p: np.array([f[p] for f in per_frame]) for p in PLANES}
    levels = np.mean([per_plane[p] for p in PLANES], axis=0)
    st = stats.offset_state(levels)
    # The state is uniform across planes (session 04 measured the step equal on all four to
    # 0.1%), so classify on the four-plane mean -- twice the precision -- and apply the labels
    # per plane.  A frame in state k has k steps of level to remove, whichever plane is read.
    near = {}
    for pl in PLANES:
        v = per_plane[pl]
        near[pl] = float(np.mean(v - st["state"] * st["separation"])) if st["separation"] else float(v.mean())
    used = {pl: float(np.mean([float(r["pedestal"]) for r in rungs
                               if int(r["gain"]) == g and r["plane"] == pl])) for pl in PLANES}
    bias[g] = dict(n=len(levels), levels=levels, per_plane=per_plane, st=st, near=near, used=used,
                   f_far=float(np.mean(st["far"])),
                   delta={pl: used[pl] - near[pl] for pl in PLANES})

print("%5s %3s %10s %10s %9s %11s %7s %8s" %
      ("gain", "n", "05 used", "near state", "delta", "step seen", "f_far", "H2 step"))
for g in GAINS:
    b = bias[g]
    sep = b["st"]["separation"]
    print("%5d %3d %10.4f %10.4f %+9.4f %11s %7.2f %8.3f" %
          (g, b["n"], b["used"]["G1"], b["near"]["G1"], b["delta"]["G1"],
           "%.4f" % sep if sep else "unresolved", b["f_far"], h2_step(g)))
print("\nresolution limit per gain (STATE_SPLIT_SIGMAS * scatter):")
print("  " + "  ".join("g%d %.4f" % (g, stats.STATE_SPLIT_SIGMAS * bias[g]["st"]["scatter"])
                       for g in GAINS))

## 3. The flat axis, and the confound that decides where it can be read

The correction is **not** simply "use the near state instead of the mixture mean", and getting that
wrong would make `g` worse rather than better. Write out what the PTC actually differenced:

```
pedestal_used   = L_near + f_b * step          mixture mean of 10 bias frames
mean_flat(rung) = S_true + L_near + f_f * step  mixture mean of 4 flats at that rung
signal          = mean_flat - pedestal_used = S_true + (f_f - f_b) * step
```

So the bias is `(f_f - f_b) * step`, and **its expectation is zero** if flats and bias hop with the
same occupancy. A mixture mean is an unbiased estimate of a mixture mean. What the state injects is
therefore not a systematic offset but **scatter, rung by rung**: `f_f` is drawn from only four
frames, so at 25% occupancy its standard deviation is about 0.22, and at gain 450 that is 2.2 counts
of noise on a signal axis whose lowest rung is 12 counts.

Removing it means putting both sides in the same state, which means classifying the flats. And
there the classifier has a rival: **the panel drifts**. L31 records repeat-to-repeat spreads of
1.79% at gain 100, and panel drift scales with the signal while the step is fixed in counts. So the
state is readable on a flat only where

```
step  >>  drift_frac * S        i.e.   S  <<  step / drift_frac
```

which makes the readable region the **low rungs** - and those are exactly the rungs the 1/var^2
weighting leans on hardest. The cell below computes that boundary per gain rather than assuming it, and marks each
plane-rung readable or not before any classification is believed. Classification itself is done on
the four-plane mean, where the shot noise halves and the state -- uniform across planes to 0.1% in
session 04 -- does not.

In [ ]:
DRIFT_FRAC = 0.0179          # L31's worst measured repeat-to-repeat spread, used as a bound
STATE_MARGIN = 3.0           # the step must beat plausible drift by this factor to be readable

rung_rows = {}
for r in rungs:
    if r["usable"] == "True":
        rung_rows.setdefault((int(r["gain"]), int(r["rung"])), {})[r["plane"]] = r

flat_dir = DATA / "session02" / "frames"
flats = {}
for (g, k), row_of in sorted(rung_rows.items()):
    paths = sorted(flat_dir.glob("flat_g%03d_r%02d_*.fits" % (g, k)))
    if len(paths) < 3:
        continue
    lv = [plane_levels(p)[0] for p in paths]
    per_plane = {pl: np.array([f[pl] for f in lv]) for pl in PLANES}
    st = stats.offset_state(np.mean([per_plane[pl] for pl in PLANES], axis=0))
    mix = float(np.mean(st["state"])) * st["separation"] if st["separation"] else 0.0
    planes = {}
    for pl, row in row_of.items():
        S = float(row["signal"])
        planes[pl] = dict(signal=S, var_pair=float(row["var_pair"]), R=float(row["R_counts"]),
                          readable=bool(h2_step(g) > STATE_MARGIN * DRIFT_FRAC * S),
                          mix_offset=mix)
    flats[(g, k)] = dict(st=st, planes=planes, f_far=float(np.mean(st["far"])))

print("%5s %7s %9s %12s %12s %10s" %
      ("gain", "rungs", "H2 step", "readable S<", "readable", "mean f_far"))
for g in GAINS:
    rows_g = [(k, f) for (gg, k), f in flats.items() if gg == g]
    pairs = [(pl, e) for _, f in rows_g for pl, e in f["planes"].items()]
    print("%5d %7d %9.3f %12.1f %8d/%-4d %10.2f" %
          (g, len(rows_g), h2_step(g), h2_step(g) / (STATE_MARGIN * DRIFT_FRAC),
           sum(e["readable"] for _, e in pairs), len(pairs),
           float(np.mean([f["f_far"] for _, f in rows_g]))))

n_read = sum(e["readable"] for f in flats.values() for e in f["planes"].values())
n_all = sum(len(f["planes"]) for f in flats.values())
print("\n%d of %d usable plane-rungs are readable for state; the rest are drift-limited"
      % (n_read, n_all))

## 4. Refitting `g`, three ways, so the change can be attributed

Session 02's rule 3: the slope of pair-difference variance against signal, weighted `1/var^2`, with
the intercept **fixed** at session 01's `R^2`. In counts that model is

```
var_pair = signal / g  +  R^2
```

which the published table satisfies exactly - `12.040 / 9.454 + 0.660^2 = 1.709` against a published
`var_pair` of 1.709 - so the refit below is the same estimator on a repaired signal axis, not a new
one. The variance axis is untouched: a uniform level cancels in a pair difference, which is why
`R` survived session 04 and why it survives this notebook too.

Three fits, so that the movement can be attributed rather than merely reported:

- **as published** - reproduces `system_gain` from the table, and is the check that the estimator
  here is the estimator there;
- **bias-side only** - the near-state pedestal replaces the mixture mean. This is the fit that would
  be *wrong* if section 3's algebra holds, and it is run precisely to show how wrong: it removes
  `f_b * step` from the signal axis while leaving `f_f * step` in place;
- **both sides** - flats and bias reduced to the same state, on the rungs section 3 called readable,
  and left alone where drift makes classification untrustworthy.

In [ ]:
def fit_g(points):
    'points: (signal, var_pair, R).  Weighted slope of (var-R^2) on signal, origin-fixed.'
    S = np.array([p[0] for p in points]); V = np.array([p[1] for p in points])
    Rr = np.array([p[2] for p in points])
    y, w = V - Rr ** 2, 1.0 / V ** 2
    slope = float(np.sum(w * S * y) / np.sum(w * S * S))     # y = S/g, so slope = 1/g
    return 1.0 / slope


def signals(g, mode, pl):
    out = []
    for (gg, k), f in flats.items():
        if gg != g or pl not in f["planes"]:
            continue
        e = f["planes"][pl]
        S = e["signal"]
        if mode in ("bias", "both"):
            S += bias[g]["delta"][pl]              # undo the mixture-mean pedestal
        if mode == "both" and e["readable"]:
            S -= e["mix_offset"]                   # and the flats' own mixture offset
        out.append((S, e["var_pair"], e["R"]))
    return out


# session 02 publishes the MEAN OVER THE FOUR CFA PLANES (ptc_gain.csv holds the per-plane
# values, which spread 2.02% at worst).  Fitting one plane and comparing it to that mean would
# charge the state for a plane-to-plane spread that has nothing to do with it -- which is what
# the reproduction check at the foot of this cell exists to catch.
fit_planes = lambda g, mode: float(np.mean([fit_g(signals(g, mode, pl)) for pl in PLANES]))


refit = {}
print("%5s %12s %12s %12s %10s %10s" %
      ("gain", "published", "as published", "bias-side", "both sides", "move %"))
for g in GAINS:
    pub = ptc_c["system_gain"]["value"][str(g)]
    a, b, c = (fit_planes(g, m) for m in ("as", "bias", "both"))
    refit[g] = dict(published=pub, reproduced=a, bias_side=b, both=c,
                    move_pct=100 * (c - pub) / pub)
    print("%5d %12.5f %12.5f %12.5f %12.5f %9.2f%%" % (g, pub, a, b, c, refit[g]["move_pct"]))

worst = max(refit, key=lambda g: abs(refit[g]["move_pct"]))
print("\nlargest move: gain %d, %+.2f%% (%.5f -> %.5f e-/ADU)"
      % (worst, refit[worst]["move_pct"], refit[worst]["published"], refit[worst]["both"]))
print("reproduction check (as-published vs published), worst: %.3f%%"
      % max(100 * abs(refit[g]["reproduced"] - refit[g]["published"]) / refit[g]["published"]
            for g in GAINS))

## 5. Gain 100, published through the classifier this time

H2 predicts a step this size at gain 100 that session 04's own night could not resolve - its limit
there was 0.776 counts against a prediction of 0.488. `results/pedestal_drift.csv` is session 01's
**450 bias frames at gain 100**, shot on 2026-08-28 to ask whether the pedestal drifts. It is out of
sample in three ways at once: a different night, a different notebook, and a gain the measuring
night could not resolve.

D71 is the rule this cell obeys: **the published classifier, not a gap.** An edge-to-edge gap
between clusters is biased low by both tails, which is how a 5.1% agreement was first reported as
0.6%. `stats.offset_state` estimates the separation between cluster *centres*, which is the quantity
H2 predicts.

In [ ]:
d_level = np.array([float(r["pedestal"]) for r in drift])
d_state = stats.offset_state(d_level)
pred100 = h2_step(100)

print("gain 100, %d frames from pedestal_drift.csv" % d_level.size)
print("  separation   %.4f counts   (H2 predicts %.4f, %+.1f%%)"
      % (d_state["separation"], pred100, 100 * (d_state["separation"] / pred100 - 1)))
print("  scatter      %.4f    threshold %.4f" % (d_state["scatter"], d_state["threshold"]))
print("  occupancy    %.3f     worst_steps %.2f (a third state would exceed 1)"
      % (float(np.mean(d_state["far"])), d_state["worst_steps"]))
print("  states found %d at %s" % (len(d_state["centres"]),
                                   np.round(d_state["centres"], 4).tolist()))

# What the mixture does to the drift rate session 01 published from these same frames.
rate = bias_c["pedestal_drift_rate"]
t = np.array([float(r["elapsed_s"]) for r in drift]) / 60.0
near = d_level - d_state["state"] * d_state["separation"]
sl_raw = np.polyfit(t, d_level, 1)[0]
sl_near = np.polyfit(t, near, 1)[0]
print("\ndrift rate, counts/min:  published %+.5f +/- %.5f" % (rate["value"], rate["uncertainty"]))
print("  raw levels    %+.5f   residual sd %.4f" % (sl_raw, np.std(d_level - np.polyval(np.polyfit(t, d_level, 1), t))))
print("  state-removed %+.5f   residual sd %.4f" % (sl_near, np.std(near - np.polyval(np.polyfit(t, near, 1), t))))

### Can the state be L31's mechanism?

`10` section 7 offered the state as a named candidate for L31 - gain 100 not repeating while gain
200 did - and was careful to call it a lead rather than a harvest. It is testable now, from
published numbers alone, and it fails on two independent counts. Both are arithmetic, and the cell
below does them rather than asserting them.

The first is a **ratio** test and needs no knowledge of the rung levels at all: L31 contrasts two
gains, and H2 says what the step is at each.

In [ ]:
L31 = dict(gain100_spread_pct=1.79, gain200_spread_pct=0.011, worst_rung_pct=5.5)

ratio_observed = L31["gain100_spread_pct"] / L31["gain200_spread_pct"]
ratio_state = h2_step(100) / h2_step(200)
print("L31 contrasts gain 100 against gain 200:")
print("  observed spread ratio      %.0fx" % ratio_observed)
print("  ratio the state predicts   %.2fx  (steps %.3f and %.3f counts)"
      % (ratio_state, h2_step(100), h2_step(200)))
print("  -> the mechanism predicts the two gains behave the SAME; L31 reports %.0fx" % ratio_observed)

# Second, independent test: magnitude.  What rung level would make a gain-100 step reach 1.79%?
S_needed = h2_step(100) / (L31["gain100_spread_pct"] / 100)
print("\nfor the step to BE 1.79%% of a rung, that rung must sit at %.1f counts" % S_needed)
print("  a linearity ladder runs 50-115% of saturation, i.e. thousands of counts;")
print("  at 2000 counts the step is %.4f%% - two orders of magnitude short."
      % (100 * h2_step(100) / 2000))

## 6. The dark bound: testing what actually limits it

`dark_current_bound` is published as an upper bound with the reason stated: *"What limits it is the
offset state, not statistics."* That sentence is a claim, and it has never been tested. Two readings
of it make different predictions, and `dark_blocks.csv` can tell them apart.

- **Random hopping.** Frames within a block land in different states. Then blocks scatter *within*
  an exposure group, and session 03's rejection should have removed it. `n_anomalous` is 0 in every
  block and the 300 s blocks agree with each other to about 0.008 counts, so this reading is already
  refuted by the published table.
- **A systematic offset that tracks exposure length.** Then every block at one exposure sits in one
  state, blocks at another exposure sit in another, and **the peer-group rule cannot see it by
  construction** - a peer group is frames sharing a setting, and exposure is a setting. This is the
  300 s / 600 s confound session 03 raised and session 04 put back in scope.

The test: a real dark current gives `excess = a + b*t` with `b > 0`. Allow each exposure group an
integer number of steps - integers only, and the step is H2's own prediction at gain 250, not a
fitted quantity - and ask whether any assignment linearises the fourteen blocks. Only *relative*
assignments are identifiable, so the search below reports the degeneracy rather than hiding it.

In [ ]:
t_b = np.array([float(r["exptime"]) for r in blocks])
ex_b = np.array([float(r["excess"]) for r in blocks])
step250 = h2_step(250)
groups = sorted(set(t_b.tolist()))
g250 = ptc_c["system_gain"]["value"]["250"]

print("excess by exposure group (counts):")
for gr in groups:
    m = t_b == gr
    print("  %6.1f s  n=%2d  mean %+.4f  sd %.4f" % (gr, m.sum(), ex_b[m].mean(), ex_b[m].std()))

results_fit = []
for ks in itertools.product(range(-2, 3), repeat=len(groups)):
    adj = ex_b + np.array([ks[groups.index(x)] for x in t_b]) * step250
    b, a = np.polyfit(t_b, adj, 1)
    rms = float(np.sqrt(np.mean((adj - (a + b * t_b)) ** 2)))
    results_fit.append((rms, ks, b, a))
results_fit.sort()

print("\n%-18s %8s %12s %11s %14s" % ("steps per group", "rms", "slope c/s", "intercept", "D e-/px/s"))
for rms, ks, b, a in results_fit[:5]:
    print("%-18s %8.4f %12.3e %11.4f %14.3e" % (str(ks), rms, b, a, b * g250))

best_rms, best_ks, best_b, best_a = results_fit[0]
tied = [r for r in results_fit if abs(r[0] - best_rms) < 1e-9]
print("\n%d assignments tie at rms %.4f - they differ by a global offset the intercept absorbs,"
      % (len(tied), best_rms))
print("so only the RELATIVE pattern is identifiable:",
      [tuple(k - tied[0][1][0] for k in r[1]) for r in tied][:1])
print("\nD under the best assignment: %.3e e-/px/s" % (best_b * g250))
print("session 03's published upper bound: %.3e e-/px/s" % dark_c["dark_current_bound"]["value"])
print("within-group sd sets the noise floor at about %.4f counts; the fit reaches %.4f"
      % (max(ex_b[t_b == gr].std() for gr in groups if (t_b == gr).sum() > 1), best_rms))
print("\nfree parameters: slope, intercept, and %d relative integers, against %d exposure groups."
      % (len(groups) - 1, len(groups)))
print("That is under-determined, which is why the number below is published as CONDITIONAL.")

## 7. What this notebook publishes

Everything above is read back from `results/` or measured from frames in `data/`. What follows is
the only cell that writes, and it writes to **this notebook's own file** - `system_gain` in
`ptc_constants.json` is left exactly as session 02 published it. A corrected constant that
overwrites its predecessor before anyone has reviewed the correction is a worse failure than a
constant that is briefly known to be wrong in a documented way.

Three of the entries below are values, and three are verdicts. `dark_current_conditional` is the
one to read most carefully: it is not a replacement for `dark_current_bound`, it is a statement
that the bound's stated *reason* is wrong and that the exposure ladder session 03 shot cannot
distinguish the alternatives on its own.

In [ ]:
PROV = dict(source_frames=len(list((DATA / "session02" / "frames").glob("*.fits"))) + len(drift) + len(blocks),
            measured_on="2026-09-04", notebook="11_state_repair.ipynb")

constants = {
  "ptc_pedestal_mixture_offset": dict(
      value={str(g): round(float(np.mean(list(bias[g]["delta"].values()))), 4) for g in GAINS},
      unit="ADC counts (pedestal session 02 used, minus its near-state level)",
      uncertainty={str(g): round(bias[g]["st"]["scatter"], 5) for g in GAINS},
      note="ten bias frames per gain through stats.offset_state.  A mixture mean sits f_far*step "
           "above the near state, so this is f_far*step and not an error in the mean itself -- "
           "section 3 is why subtracting it alone would make g worse, not better",
      **PROV),

  "ptc_bias_occupancy": dict(
      value={str(g): round(bias[g]["f_far"], 4) for g in GAINS},
      unit="fraction of frames far, session 02's bias groups",
      uncertainty=None,
      note="a third independent night at these settings.  Session 04 measured 0.333 at gain 0 and "
           "0.250 at gain 450; notebook 10 backed 0.200 out of session 01's published pedestal",
      **PROV),

  "system_gain_restated": dict(
      value={str(g): round(refit[g]["both"], 5) for g in GAINS},
      unit="e- per ADC count",
      uncertainty={str(g): round(abs(refit[g]["both"] - refit[g]["bias_side"]), 5) for g in GAINS},
      note="session 02's estimator on a signal axis with flats and bias reduced to the same state, "
           "on the rungs section 3 found readable against panel drift.  The uncertainty field is "
           "the spread between the bias-side-only and both-sides fits, which is what the "
           "drift-limited rungs leave open.  It does NOT supersede ptc_constants.json's "
           "system_gain: that is a record decision, not a notebook's",
      **PROV),

  "system_gain_move_pct": dict(
      value={str(g): round(refit[g]["move_pct"], 3) for g in GAINS},
      unit="percent change from session 02's published system_gain",
      uncertainty=None,
      note="what the state cost g at each gain.  Session 02's own fit residual is 1.344%, so a "
           "move below that is inside the noise the gain law already carries",
      **PROV),

  "state_step_at_gain100": dict(
      value=round(float(d_state["separation"]), 4),
      unit="ADC counts",
      uncertainty=round(float(d_state["scatter"]), 5),
      note="stats.offset_state over session 01's 450-frame pedestal_drift block, out of sample in "
           "three ways: a different night, a different notebook, and a gain session 04 could not "
           "resolve (its limit there was 0.776 against a 0.488 prediction).  H2 predicts %.4f, "
           "agreeing to %+.1f%%.  Measured with the published classifier and not with an "
           "edge-to-edge gap (D71)" % (pred100, 100 * (d_state["separation"] / pred100 - 1)),
      **PROV),

  "state_explains_L31": dict(
      value=False,
      unit="boolean",
      uncertainty=None,
      note="L31 contrasts a 1.79%% repeat spread at gain 100 with 0.011%% at gain 200, a factor of "
           "%.0f.  H2's steps at those gains are %.3f and %.3f counts, a factor of %.2f: the "
           "mechanism predicts the two behave the same.  Independently, 1.79%% of a linearity rung "
           "near 2000 counts is ~36 counts against a half-count state.  L31 stays queued with its "
           "mechanism unknown; what IS confirmed is the bimodality itself"
           % (ratio_observed, h2_step(100), h2_step(200), ratio_state),
      **PROV),

  "dark_bound_limited_by": dict(
      value="exposure-correlated offset, not random hopping",
      unit="one of: random hopping, exposure-correlated offset, statistics",
      uncertainty=None,
      note="session 03 rejected 0 anomalous frames in every block and its 300 s blocks agree to "
           "0.008 counts, so within-group hopping is refuted by its own published table.  What "
           "survives is an offset that tracks exposure length, which a peer-group rule cannot see "
           "because exposure is what defines a peer group",
      **PROV),

  "dark_current_conditional": dict(
      value=round(float(best_b * g250), 8),
      unit="e-/px/s at -10 C, CONDITIONAL on the best integer-step assignment",
      uncertainty=None,
      note="excess against exposure with each group allowed an integer number of H2 steps: the "
           "assignment %s linearises 14 blocks to %.4f counts rms, at the within-group noise "
           "floor.  It is NOT a replacement for dark_current_bound: slope, intercept and %d "
           "relative integers against %d exposure groups is under-determined.  The decisive test "
           "is a fifth and sixth exposure, which is a bench night"
           % (str(best_ks), best_rms, len(groups) - 1, len(groups)),
      **PROV),
}

CONSTANTS = RESULTS / "state_repair_constants.json"
with open(CONSTANTS, "w", encoding="utf8") as fh:
    json.dump(constants, fh, indent=2)
print(f"wrote {CONSTANTS} with {len(constants)} constants, provenance on each")
for k, v in constants.items():
    print("  %-30s %s" % (k, v["value"] if not isinstance(v["value"], dict) else "per gain"))

## 8. What is settled, and what the next session inherits

This notebook is the measuring half. `12` reads these files back and explains what they mean for
someone deciding what to do next; if the two ever disagree, `results/` is right and `12` is the bug.

**Two LEGACY entries were opened and neither is drained.** L31 is *narrowed*, not verified: its
bimodality is now published with provenance, and the state is ruled out as its mechanism, so it goes
to the linearity session with a smaller question and one candidate fewer. That is a change to the
entry, not a deletion of it - and `CLAUDE.md` is explicit that an entry leaves only when the claim
it carries has been checked.

**One thing the notebook deliberately did not do.** It did not overwrite `system_gain`. The two
files now hold different numbers for the same quantity, which is a trap if it is left standing and
a decision if it is taken - and taking it is the first item for review.